<a href="https://colab.research.google.com/github/patrick-nelissen/PickleballMixer/blob/main/PickleballMIxer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#@title Mixer Round Robin
import random
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define the default players
names = 'Saskia, Rubi, Teresa, Nicole, Amy, Mike, Patrick, Babara, Cristi, Marlee, Keeley, Mai, Linda, Heidi, David, Dana, Elana, Sarah, Jon, Sue, Sierra, Cary'

full_sets = len([n.strip() for n in names.split(',')])//4

# Create the text area widget
text_area = widgets.Textarea(
    value=names,
    description='Players:',
    disabled=False,
    layout=widgets.Layout(flex_flow="column", align_items="flex-start", width='100%', height='200px')  # Set custom width and height
)

# Define the number of courts
courts_slider = widgets.IntSlider(
    value=full_sets,
    min=0,
    max=10,
    step=1,
    description='# of Courts:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Define the number of rounds
rounds_slider = widgets.IntSlider(
    value=8,
    min=0,
    max=10,
    step=1,
    description='# of rounds:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Random seed factor
seed_slider = widgets.IntSlider(
    value=42,
    min=40,
    max=45,
    step=1,
    description='Seed #:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Create a button to submit the form
submit_button = widgets.Button(
    description='Create round robins',
    button_style='success', # Choices: 'success', 'info', 'warning', 'danger', or ''
    tooltip='Click to create round robins'
)

# Create an output widget to display results cleanly
output_area = widgets.Output()

def generate_pickleball_mixer(players, num_courts, num_rounds, seed=None):
  """Generates a rotating-doubles pickleball round-robin schedule.

    Parameters:
        players (list): List of player names.
        num_courts (int): Number of available courts.
        num_rounds (int): Total number of rounds to play.
        seed (int, optional): Random seed for reproducible schedules.

    Returns:
        list: A list of rounds, where each round contains a list of court assignments.
    """
  if seed is not None:
    random.seed(seed)

  player_pool = list(players)
  schedule = []
  sit_out_counts = {p: 0 for p in players}

  for r in range(1, num_rounds + 1):
    round_data = {"round": r, "matches": [], "sitting_out": []}

    # Determine how many players can play vs. need to sit out
    max_players_on_court = num_courts * 4
    shuffled_players = player_pool[:]

    # Players who have sat out the MOST get priority to play;
    # ties are broken randomly so byes are still random among equals
    shuffled_players.sort(key=lambda p: (-sit_out_counts[p], random.random()))

    active_players = shuffled_players[:max_players_on_court]
    sitting_out = shuffled_players[max_players_on_court:]

    for p in sitting_out:
      sit_out_counts[p] += 1
      round_data["sitting_out"].append(p)

    # Shuffle active players to randomize partnerships
    random.shuffle(active_players)

    matches = []
    for c in range(num_courts):
      court_idx = c * 4
      if court_idx + 4 <= len(active_players):
        court_group = active_players[court_idx : court_idx + 4]
        match = {
            "court": c + 1,
            "team_1": (court_group[0], court_group[1]),
            "team_2": (court_group[2], court_group[3]),
        }
        matches.append(match)

    round_data["matches"] = matches
    schedule.append(round_data)

  return schedule


# Define what happens when the button is clicked
def on_submit_clicked(b):

  names = text_area.value
  # Parse the comma-separated string into a clean Python list
  players_list = [name.strip() for name in names.split(",") if name.strip()]

  courts = courts_slider.value
  rounds = rounds_slider.value

  with output_area:
    clear_output() # Clears previous results from the container

    names = text_area.value

    # Parse the comma-separated string into a clean Python list
    players_list = [name.strip() for name in names.split(",") if name.strip()]

    if not names.strip() or (len(players_list) < 4):
        print("⚠️ Please enter at least 4 players before submitting.")
    else:
        print("\n")

        seednmbr = seed_slider.value
        courts = courts_slider.value
        rounds = rounds_slider.value

        schedule = generate_pickleball_mixer(
            players=players_list, num_courts=courts, num_rounds=rounds, seed=seednmbr
        )

        for r_info in schedule:
          print(f"\n--- Round {r_info['round']} ---")
          for m in r_info["matches"]:
            t1 = " & ".join(m["team_1"])
            t2 = " & ".join(m["team_2"])
            print(f"  Court {m['court']}: [{t1}] vs [{t2}]")

          if r_info["sitting_out"]:
            print(f"  Sitting out (Bye): {', '.join(r_info['sitting_out'])}")




# Bind the click event to the function
submit_button.on_click(on_submit_clicked)

# Arrange and display the form UI
form_layout = widgets.VBox([courts_slider, rounds_slider, seed_slider, text_area, submit_button, output_area])
display(form_layout)